## Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output

## Pydantic

Pydantic models provide the richest feature set with field validation, descriptions and nested structures

In [1]:
import os
from langchain.chat_models import init_chat_model
model=init_chat_model("groq:qwen/qwen3.6-27b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x00000212FCD86EA0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000212FBAD4BC0>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [3]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x00000212FCD86EA0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000212FBAD4BC0>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'

In [4]:
model.invoke("Provide details about the movie The Prestige")

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Request**: The user asks for "details about the movie The Prestige". This is a straightforward request for information about a specific film. I need to provide comprehensive, accurate, and well-structured details.\n\n2.  **Identify Key Information Needed**:\n   - Title: The Prestige\n   - Release Year: 2006\n   - Director: Christopher Nolan\n   - Screenplay: Jonathan Nolan & Christopher Nolan\n   - Based on: Novel by Christopher Priest\n   - Genre: Mystery, Drama, Thriller, Period Piece\n   - Cast: Hugh Jackman, Christian Bale, Michael Caine, Scarlett Johansson, Rebecca Hall, David Bowie, Andy Serkis\n   - Plot Summary: Two rival magicians in late 19th-century London engage in a dangerous battle of one-upmanship, leading to obsession, betrayal, and dark secrets.\n   - Themes: Obsession, sacrifice, illusion vs. reality, rivalry, the cost of greatness\n   - Structure: Non-linear narrative, dual POV (Borden

In [5]:
model_with_structure.invoke("Provide details about the movie The Prestige")

Movie(title='The Prestige', year=2006, director='Christopher Nolan', rating=8.5)

## Message output alongside parsed structure

In [11]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details"""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="This year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movies rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie The Prestige")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "Provide details about the movie The Prestige"\n   - I need to use the `Movie` function to provide details about this movie.\n   - Required parameters for `Movie`: title, year, director, rating\n   - I know the movie "The Prestige" (2006) directed by Christopher Nolan, based on Christopher Priest\'s novel, starring Hugh Jackman and Christian Bale.\n   - I need to verify the exact year, director, and rating.\n   - Year: 2006\n   - Director: Christopher Nolan\n   - Rating: I should provide a reasonable rating out of 10. The Prestige has an IMDb rating of around 8.5/10. I\'ll use 8.5.\n   - All required parameters are available.\n\n2.  **Construct Function Call:**\n   - title: "The Prestige"\n   - year: 2006\n   - director: "Christopher Nolan"\n   - rating: 8.5\n\n3.  **Execute Function Call:**\n   - Call the `Movie` function with the specified

## Nested Structure

In [ ]:
class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie The Prestige")
response

    

MovieDetails(title='The Prestige', year=2006, cast=[Actor(naem='Hugh Jackman', role='Robert Angier'), Actor(naem='Christian Bale', role='Alfred Borden'), Actor(naem='Scarlett Johansson', role='Olivia Wenscombe'), Actor(naem='Michael Caine', role='Cutter'), Actor(naem='Rebecca Hall', role='Sarah Borden')], genres=['Thriller', 'Drama', 'Mystery', 'Sci-Fi'], budget=40.0)